# Batch retrieval in naive search — a benchmark

`NaiveSearchEngine.search()` is a one-line wrapper over `batch_search()`:

```python
async def search(self, query, params=None):
    return (await self.batch_search([query], params))[0]
```

Everything in RAGU is async, so the obvious way to "parallelize" 5 000 queries is
to fire them all through `asyncio.gather`. This notebook measures whether that
actually helps, against real batching.

**Spoiler:** it does not. `asyncio.gather` is *concurrency*, not *parallelism*.
The retrieval path here is CPU-bound numpy running on the event loop thread, and
there is nothing for the loop to interleave — so 5 000 gathered `search()` calls
land within noise of a plain for-loop. Only `batch_search` changes the amount of
work done, and it is about 10x faster than either.

**Workload**

| | |
|---|---|
| vectors in the index | 50 000 |
| queries | 5 000 |
| batch size | 512 |
| `top_k` | 10 |
| embedding dimension | 384 |
| vector store | `NanoVectorDBStorage` (the default) |

**What is measured:** the retrieval path only — query encoding, the vector search,
and the chunk lookup that resolves hits into text.

**What is deliberately excluded:** the embedding API and answer generation. Both are
network-bound and would dominate the clock, hiding the thing being measured. The
notebook uses a local stub embedder, so every number below is CPU work inside RAGU.
The last section covers what changes when the embedder is real — it is the case
where `gather` finally earns its keep, and it still loses to batching.

This notebook needs **no API keys** and takes about a minute.

In [ ]:
import asyncio
import shutil
import tempfile
import time

import numpy as np

from ragu.common.global_parameters import Settings
from ragu import BuilderArguments, KnowledgeGraph, NaiveSearchEngine, StorageArguments
from ragu.models.embedder import Embedder
from ragu.search_engine.naive_search import NaiveSearchParams

# Must be set before any storage is constructed.
WORKDIR = tempfile.mkdtemp(prefix="ragu_batch_benchmark_")
Settings.language = "english"
Settings.storage_folder = WORKDIR

N_VECTORS = 50_000
N_QUERIES = 5_000
BATCH_SIZE = 512
TOP_K = 10
DIM = 384
N_TOPICS = 200

print(f"workdir: {WORKDIR}")

## The stub embedder

Uniformly random vectors would be a bad model of a corpus: in 384 dimensions they
are all near-orthogonal, every cosine score lands around zero, and the retrieved
sets of different queries never overlap. Real corpora are clumpy.

So vectors are drawn around `N_TOPICS` anchor directions with noise. That gives
realistic score magnitudes and a realistic amount of overlap between what different
queries retrieve.

In [ ]:
class TopicEmbedder(Embedder):
    """
    Deterministic local embedder producing clustered unit vectors.

    :param dim: Embedding dimensionality.
    :param n_topics: Number of anchor directions vectors are drawn around.
    :param seed: Seed for reproducible runs.
    """

    def __init__(self, dim: int = DIM, n_topics: int = N_TOPICS, seed: int = 0) -> None:
        self._dim = dim
        self._rng = np.random.default_rng(seed)
        anchors = self._rng.standard_normal((n_topics, dim)).astype(np.float32)
        self._anchors = anchors / np.linalg.norm(anchors, axis=1, keepdims=True)

    @property
    def dim(self) -> int:
        return self._dim

    def _vectors(self, count: int) -> np.ndarray:
        index = self._rng.integers(0, len(self._anchors), size=count)
        noise = self._rng.standard_normal((count, self._dim)).astype(np.float32) * 0.35
        vectors = self._anchors[index] + noise
        return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

    async def embed_text(self, text: str, **kwargs):
        return list(self._vectors(1)[0])

    async def batch_embed_text(self, texts: list[str], desc: str | None = None, **kwargs):
        return list(self._vectors(len(texts)))


embedder = TopicEmbedder()
print(f"dim={embedder.dim}, anchors={N_TOPICS}")

## Build the index

`build_only_vector_context=True` skips entity and relation extraction entirely —
no LLM is involved, which is exactly what naive search needs. With `chunker=None`
each document becomes one chunk, so 50 000 documents give 50 000 vectors.

`score_threshold=0.0` matters: `NanoVectorDBStorage` defaults to `0.2` for cosine
and silently drops weaker hits. Leaving the default would return partial result
sets and quietly make the benchmark measure less work than `top_k` implies.

In [ ]:
knowledge_graph = KnowledgeGraph(
    llm=None,
    embedder=embedder,
    chunker=None,
    builder_settings=BuilderArguments(build_only_vector_context=True),
    storage_settings=StorageArguments(vdb_storage_kwargs={"score_threshold": 0.0}),
)

documents = [f"Document {i}: synthetic corpus text for retrieval benchmarking." for i in range(N_VECTORS)]

started = time.perf_counter()
await knowledge_graph.build_from_docs(documents)
build_seconds = time.perf_counter() - started

stored = len(await knowledge_graph.index.chunks_kv_storage.all_keys())
print(f"built {stored} chunks in {build_seconds:.1f}s")

In [ ]:
engine = NaiveSearchEngine(llm=None, knowledge_graph=knowledge_graph, embedder=embedder)
params = NaiveSearchParams(top_k=TOP_K)
queries = [f"query {i} about the synthetic corpus" for i in range(N_QUERIES)]

# Warm-up: the first call pays for lazy numpy/BLAS setup, and a cold measurement
# would be charged to whichever mode happens to run first.
warmup = await engine.batch_search(queries[:16], params)
retrieved = sum(len(result.result.chunks) for result in warmup)
print(f"sanity check: {retrieved} chunks for 16 queries (expected {16 * TOP_K})")

## The four modes

All four answer the same 5 000 queries and return the same results. They differ
only in how the calls are arranged.

In [ ]:
async def sequential() -> None:
    """One `search()` at a time — the naive reference point."""
    for query in queries:
        await engine.search(query, params)


async def gather_all() -> None:
    """All 5 000 `search()` coroutines handed to one `asyncio.gather`."""
    await asyncio.gather(*[engine.search(query, params) for query in queries])


async def gather_waves() -> None:
    """`asyncio.gather` in waves of BATCH_SIZE — bounded concurrency."""
    for offset in range(0, N_QUERIES, BATCH_SIZE):
        wave = queries[offset:offset + BATCH_SIZE]
        await asyncio.gather(*[engine.search(query, params) for query in wave])


async def batched() -> None:
    """`batch_search()` with BATCH_SIZE queries per call — real batching."""
    for offset in range(0, N_QUERIES, BATCH_SIZE):
        await engine.batch_search(queries[offset:offset + BATCH_SIZE], params)


REPEATS = 2


async def measure(label: str, mode) -> float:
    """
    Time a mode `REPEATS` times and keep the best run.

    Minimum-of-N rather than the mean: a single run picks up whatever else the
    machine was doing, and those interruptions only ever make a run slower. The
    three non-batched modes here land close enough together that one noisy run
    reorders them.

    :param label: Name printed with the result.
    :param mode: Zero-argument coroutine function to time.
    :returns: Best observed wall-clock time in seconds.
    """
    timings = []
    for _ in range(REPEATS):
        started = time.perf_counter()
        await mode()
        timings.append(time.perf_counter() - started)
    best = min(timings)
    spread = (max(timings) - best) / best * 100
    print(f"{label:<38} {best:7.2f}s {N_QUERIES / best:9.1f} q/s   (spread {spread:4.1f}%)")
    return best

## Run them

The three non-batched modes take roughly 20 seconds per repeat; the batched one is
the fast outlier. Total runtime for this cell is a couple of minutes.

In [ ]:
sequential_seconds = await measure("sequential for-loop", sequential)
gather_seconds = await measure(f"asyncio.gather, all {N_QUERIES} at once", gather_all)
waves_seconds = await measure(f"asyncio.gather in waves of {BATCH_SIZE}", gather_waves)
batched_seconds = await measure(f"batch_search of {BATCH_SIZE}", batched)

## Result

In [ ]:
print(f"{'mode':<34} {'calls':>7} {'seconds':>9} {'q/s':>10} {'vs seq':>8}")
print("-" * 71)
for label, calls, seconds in (
    ("sequential for-loop", N_QUERIES, sequential_seconds),
    ("asyncio.gather (all at once)", N_QUERIES, gather_seconds),
    (f"asyncio.gather (waves of {BATCH_SIZE})", N_QUERIES, waves_seconds),
    (f"batch_search ({BATCH_SIZE})", -(-N_QUERIES // BATCH_SIZE), batched_seconds),
):
    print(f"{label:<34} {calls:>7} {seconds:>9.2f} {N_QUERIES / seconds:>10.1f} "
          f"{sequential_seconds / seconds:>7.2f}x")
print("-" * 71)
print(f"batch_search vs asyncio.gather: {gather_seconds / batched_seconds:.1f}x")

## Why `asyncio.gather` does nothing here

`gather` does not make code parallel. It schedules coroutines on **one** event loop
thread and switches between them whenever one of them *suspends* — at real I/O, at
`asyncio.sleep`, at anything that yields control back to the loop.

Follow what a single `search()` does in this benchmark:

1. `build_query_vectors` → the stub embedder, pure numpy, never yields.
2. `chunks_vector_db.query` → `NanoVectorDBStorage`, a numpy matrix multiply,
   never yields.
3. `chunks_kv_storage.get_by_ids` → `JsonKVStorage`, an in-memory dict lookup,
   never yields.

There is not a single genuine suspension point in the whole path. The coroutines
therefore run strictly one after another, exactly like the for-loop, with the added
cost of creating and scheduling 5 000 tasks.

That is exactly what the table shows: the three non-batched modes land **within
noise of each other**. Which of them comes out marginally ahead flips between runs
and between machines — during development of this notebook `gather` was 15% slower
than the loop on one run and 20% faster on the next. Do not read a winner into that
gap; read the absence of one. None of the three changes the amount of work, so none
of them can be meaningfully faster.

Only the fourth row moves.

`batch_search` wins for a completely different reason: it does not rearrange the
calls, it **reduces the work**. 512 separate `(1 × 384) @ (384 × 50000)` products
become one `(512 × 384) @ (384 × 50000)` product that BLAS can actually parallelize
across cores — plus one deduplicated KV lookup instead of 512.

## Where the time goes

`GraphRetriever.query_chunks` does three things per batch. Timing them separately
shows what batching is up against.

In [ ]:
vector_db = knowledge_graph.index.chunks_vector_db
kv_storage = knowledge_graph.index.chunks_kv_storage
batch = queries[:BATCH_SIZE]

started = time.perf_counter()
points = await engine.retriever.build_query_vectors(batch)
encode_ms = (time.perf_counter() - started) * 1000

started = time.perf_counter()
hits = await vector_db.query(points, top_k=TOP_K)
search_ms = (time.perf_counter() - started) * 1000

unique_ids = list({hit.id for query_hits in hits for hit in query_hits})
started = time.perf_counter()
await kv_storage.get_by_ids(unique_ids)
lookup_ms = (time.perf_counter() - started) * 1000

total_ms = encode_ms + search_ms + lookup_ms
print(f"one batch of {BATCH_SIZE}:")
for label, value in (("encode queries", encode_ms), ("vector search", search_ms), ("chunk lookup", lookup_ms)):
    print(f"  {label:<16} {value:7.1f} ms  {100 * value / total_ms:5.1f}%")

The vector search dominates — the honest result for an in-process store with no
network anywhere. And that is exactly the stage batching collapses:

In [ ]:
SAMPLE = 200
started = time.perf_counter()
for point in points[:SAMPLE]:
    await vector_db.query([point], top_k=TOP_K)
per_query_ms = (time.perf_counter() - started) / SAMPLE * 1000

print(f"one at a time: {per_query_ms:.2f} ms/query -> {per_query_ms * BATCH_SIZE:.0f} ms per {BATCH_SIZE}")
print(f"batched:       {search_ms:.1f} ms per {BATCH_SIZE}")
print(f"vector-search speedup alone: {per_query_ms * BATCH_SIZE / search_ms:.1f}x")

## Deduplication before the chunk lookup

`query_chunks` collects hit ids across the *whole batch*, deduplicates them, and
issues a single KV lookup. Popular chunks retrieved by many queries are fetched
once instead of once per query.

How much this saves is entirely a property of your query distribution — large when
users ask similar things, near zero for unrelated queries. It is a rounding error
against a local `JsonKVStorage` and a real saving against a remote store.

In [ ]:
total_hits = sum(len(query_hits) for query_hits in hits)
print(f"{total_hits} hits -> {len(unique_ids)} unique chunk ids "
      f"({100 * (1 - len(unique_ids) / total_hits):.1f}% of lookups avoided)")
print(f"gather and the for-loop both issue {BATCH_SIZE} separate KV lookups "
      f"for these queries; batch_search issues 1")

## Choosing a batch size

In [ ]:
sweep_queries = queries[:2048]
print(f"{'batch_size':>11} {'seconds':>9} {'q/s':>10}")
print("-" * 32)
for size in (1, 32, 128, 512, 2048):
    started = time.perf_counter()
    for offset in range(0, len(sweep_queries), size):
        await engine.batch_search(sweep_queries[offset:offset + size], params)
    seconds = time.perf_counter() - started
    print(f"{size:>11} {seconds:>9.2f} {len(sweep_queries) / seconds:>10.1f}")

The shape to read off that table: the first step is by far the largest — going from
1 to 32 buys roughly 5x on its own. Gains keep accruing up to about 512, then the
curve flattens, with 2048 barely distinguishable from 512.

The middle of the curve is noisy between runs (32 and 128 often land within a few
percent of each other), because at those sizes the matrix multiply is already large
enough that scheduling and allocation dominate the difference. Do not read a precise
optimum out of a single run.

Practically: take the largest batch whose query vectors and hit sets fit comfortably
in memory. 512 is a reasonable default and the one this benchmark used.

## Clean up

In [ ]:
await knowledge_graph.index.close()
shutil.rmtree(WORKDIR, ignore_errors=True)
print(f"removed {WORKDIR}")